In [4]:
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_experimental.text_splitter import SemanticChunker
import os 
from dotenv import load_dotenv
load_dotenv()
from langchain_openai import OpenAIEmbeddings

In [5]:
docs = PyPDFLoader("../CH-2_RAG/NovaS.pdf").load()

full_text = "\n".join([p.page_content for p in docs])
full_text

'NovaSphere Technologies is a fictional organization created to represent a modern data \nand technology company that has grown gradually over the years. The organization was \nfounded in 2016 by a small group of software engineers who strongly believed that data \nwould become one of the most valuable assets for every business in the future. At the \nbeginning, the company did not have large investments or a big office. Instead, it started \nwith only six employees working together in a small shared workspace. The founders were \nnot focused on becoming successful overnight. Their main goal was to build strong \ntechnical knowledge, gain practical experience, and slowly grow by delivering real value to \ntheir clients. Most of the early work involved helping small companies understand their \nexisting data and use simple reporting solutions to make better business decisions. \nDuring the first year of operations, the company worked mostly with local startups that did \nnot have large 

## STEP-1 : Creating Chunks of the text data

In [6]:
chunker = SemanticChunker(
    OpenAIEmbeddings(model="text-embedding-3-small"),
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=60
)

semantic_chunks = chunker.create_documents([full_text])
semantic_chunks

[Document(metadata={}, page_content='NovaSphere Technologies is a fictional organization created to represent a modern data \nand technology company that has grown gradually over the years. The organization was \nfounded in 2016 by a small group of software engineers who strongly believed that data \nwould become one of the most valuable assets for every business in the future.'),
 Document(metadata={}, page_content='At the \nbeginning, the company did not have large investments or a big office.'),
 Document(metadata={}, page_content='Instead, it started \nwith only six employees working together in a small shared workspace. The founders were \nnot focused on becoming successful overnight.'),
 Document(metadata={}, page_content='Their main goal was to build strong \ntechnical knowledge, gain practical experience, and slowly grow by delivering real value to \ntheir clients. Most of the early work involved helping small companies understand their \nexisting data and use simple reporting 

### STEP-2&3 : Creating Embeddings and Storing them in Vector DB

In [7]:
from langchain_community.vectorstores import Chroma
embed_model = OpenAIEmbeddings(model="text-embedding-3-small")
chroma_db = Chroma.from_documents(semantic_chunks, embed_model, persist_directory="./chroma_db_semantic")

## Step-4 : Connection & Retrieval

In [8]:
chroma_db_con = Chroma(persist_directory="./chroma_db_semantic", embedding_function=embed_model)

/var/folders/cr/34qx8njd48g07ps00m4fshq40000gn/T/ipykernel_25716/2430328870.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  chroma_db_con = Chroma(persist_directory="./chroma_db_semantic", embedding_function=embed_model)


In [9]:
chroma_db_con.similarity_search("By 2019, how many employees were there?", k=3)

[Document(metadata={}, page_content='The \nnumber of employees increased again, and the company also started hiring people who \nhad experience in big data technologies. This helped the organization build stronger \ntechnical capabilities and handle larger clients. Another important milestone came in 2022 when the organization decided to invest more \ntime in research and learning new technologies. Instead of working only on client projects, \nthe company created a small internal research group.'),
 Document(metadata={}, page_content='The organization began working with \ncompanies from different industries such as retail, education, and small healthcare \nproviders. Each industry had different challenges, and this helped the team improve its \nunderstanding of real-world business problems. The company also started focusing more \non improving the quality of its work instead of simply increasing the number of projects. The founders believed that long-term growth would come only if the 

## STEP-5 : LLM Integration and Answer Generation

In [10]:
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

In [11]:
user_query = input("Enter your question: ")

rel_chunks = chroma_db_con.similarity_search(user_query, k=3)

rel_chunks_content = []
for i, chunk in enumerate(rel_chunks):
    rel_chunks_content.append(chunk.page_content)
rel_chunks_content = str(rel_chunks_content)

llm.invoke(f"{user_query}, Use the following context to answer the question: {rel_chunks_content}")

AIMessage(content='1. Encourage continuous learning: Encourage employees to learn new technologies and skills to stay updated with the latest trends in the industry.\n\n2. Foster a culture of innovation: Encourage employees to share their ideas and experiment with new technologies to drive innovation within the organization.\n\n3. Focus on communication skills: Improving communication skills can help employees effectively collaborate and work together towards common goals.\n\n4. Implement flexible working options: Offering flexible working options can help employees maintain a healthy work-life balance, leading to increased productivity and job satisfaction.\n\n5. Prioritize employee satisfaction: Investing in employee satisfaction and creating a supportive work environment can lead to a happy and motivated team, ultimately benefiting the organization in the long run.\n\n6. Build strong relationships with clients: By focusing on maintaining long-term relationships with clients, the org